# Day 19: Duplicate Records and Data Validation

**Dataset:** `Customer_Dataset.csv`

This notebook detects duplicate rows and performs simple validation checks to identify
inconsistent or invalid records.

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../datasets/Customer_Dataset.csv")

# Preview the data
df.head()

,Order ID,Customer Name,Age,Gender,City,Signup Date,Purchase Amount,Email
0,1000,Bob Lee,25.0,male,LA,01/22/2023,$45,alice@example.com
1,1001,Bob Lee,22.0,F,NaN,15-04-2023,$120.50,john@example.com
2,1002,MARY JOHNSON,-5.0,Female,miami,19-05-2023,$120.50,NaN
3,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
4,1004,David Kim,29.0,female,Chicago,01/22/2023,"1,200.00",bad-email


In [2]:
print(df.shape)
df.info()

(43, 8)
<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order ID         43 non-null     int64  
 1   Customer Name    43 non-null     str    
 2   Age              39 non-null     float64
 3   Gender           37 non-null     str    
 4   City             41 non-null     str    
 5   Signup Date      40 non-null     str    
 6   Purchase Amount  37 non-null     str    
 7   Email            31 non-null     str    
dtypes: float64(1), int64(1), str(6)
memory usage: 2.8 KB


## 2. Duplicate Record Analysis

**`duplicated()` — a True/False flag for every row that repeats an earlier row exactly**

In [3]:
print(df.duplicated().head(10))
print()
print("Total duplicate rows:", df.duplicated().sum())

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool

Total duplicate rows: 3


**Viewing just the duplicate rows themselves**

In [4]:
df[df.duplicated()]

,Order ID,Customer Name,Age,Gender,City,Signup Date,Purchase Amount,Email
40,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
41,1010,Jack Ryan,NaN,Female,Boston,2023.03.10,89.99,NaN
42,1022,Jack Ryan,45.0,NaN,miami,15-01-2023,-20,bob@example


**Using `keep=False` to see BOTH the original row and its duplicate(s), side by side**

In [5]:
df[df.duplicated(keep=False)].sort_values("Order ID")

,Order ID,Customer Name,Age,Gender,City,Signup Date,Purchase Amount,Email
3,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
40,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
10,1010,Jack Ryan,NaN,Female,Boston,2023.03.10,89.99,NaN
41,1010,Jack Ryan,NaN,Female,Boston,2023.03.10,89.99,NaN
22,1022,Jack Ryan,45.0,NaN,miami,15-01-2023,-20,bob@example
42,1022,Jack Ryan,45.0,NaN,miami,15-01-2023,-20,bob@example


**Checking for duplicates based on a specific key column (`Order ID`) instead of the whole row**

In [6]:
print("Duplicate Order IDs:", df.duplicated(subset=["Order ID"]).sum())
df[df.duplicated(subset=["Order ID"], keep=False)].sort_values("Order ID")

Duplicate Order IDs: 3


,Order ID,Customer Name,Age,Gender,City,Signup Date,Purchase Amount,Email
3,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
40,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
10,1010,Jack Ryan,NaN,Female,Boston,2023.03.10,89.99,NaN
41,1010,Jack Ryan,NaN,Female,Boston,2023.03.10,89.99,NaN
22,1022,Jack Ryan,45.0,NaN,miami,15-01-2023,-20,bob@example
42,1022,Jack Ryan,45.0,NaN,miami,15-01-2023,-20,bob@example


## 3. Cleaned Dataset Without Unwanted Duplicates

`drop_duplicates()` removes duplicate rows, keeping the **first** occurrence by default.

In [7]:
df_deduplicated = df.drop_duplicates()

print("Original row count:", len(df))
print("Row count after drop_duplicates():", len(df_deduplicated))
print("Rows removed:", len(df) - len(df_deduplicated))

Original row count: 43
Row count after drop_duplicates(): 40
Rows removed: 3


**A word of caution on `drop_duplicates()`**

`drop_duplicates()` (with no arguments) only removes rows that match **every single column**
exactly. It would NOT catch the case where the same customer appears twice with slightly
different information (a typo, a different signup date, etc.) — that kind of "near duplicate"
needs a closer look, usually by checking `subset=` on a key column like `Order ID` instead of
relying on whole-row matching alone.

In [8]:
# Confirm no full-row duplicates remain
print("Remaining duplicate rows:", df_deduplicated.duplicated().sum())
print("Remaining duplicate Order IDs:", df_deduplicated.duplicated(subset=["Order ID"]).sum())

Remaining duplicate rows: 0
Remaining duplicate Order IDs: 0


## 4. Data Validation Checks

Working from the deduplicated dataset, checking for out-of-range values and unexpected
categories.

**Check 1: `Age` should be within a realistic human range (0–120)**

In [9]:
invalid_age = df_deduplicated[(df_deduplicated["Age"] < 0) | (df_deduplicated["Age"] > 120)]
print(len(invalid_age), "rows with an invalid Age")
invalid_age[["Order ID", "Customer Name", "Age"]]

7 rows with an invalid Age


,Order ID,Customer Name,Age
2,1002,MARY JOHNSON,-5.0
7,1007,Paul Young,150.0
9,1009,Karen White,150.0
12,1012,Paul Young,-5.0
13,1013,Karen White,150.0
28,1028,Frank Miller,-5.0
31,1031,Nate Ross,-5.0


**Check 2: `Gender` should be one of the expected categories**

In [10]:
expected_genders = {"m", "male", "f", "female"}
gender_lower = df_deduplicated["Gender"].str.strip().str.lower()

invalid_gender = df_deduplicated[~gender_lower.isin(expected_genders) & gender_lower.notna()]
print(len(invalid_gender), "rows with an unexpected Gender value")
print("Actual unique values found:", df_deduplicated["Gender"].unique())

0 rows with an unexpected Gender value
Actual unique values found: <StringArray>
['male', 'F', 'Female', 'M', 'female', 'Male', nan]
Length: 7, dtype: str


**Check 3: `Purchase Amount` should not be negative**

In [11]:
def to_number(value):
    if pd.isna(value):
        return None
    return float(str(value).replace("$", "").replace(",", ""))


purchase_numeric = df_deduplicated["Purchase Amount"].apply(to_number)
invalid_purchase = df_deduplicated[purchase_numeric < 0]
print(len(invalid_purchase), "rows with a negative Purchase Amount")
invalid_purchase[["Order ID", "Customer Name", "Purchase Amount"]]

5 rows with a negative Purchase Amount


,Order ID,Customer Name,Purchase Amount
5,1005,Bob Lee,-20
9,1009,Karen White,-20
13,1013,Karen White,-20
19,1019,John Smith,-20
22,1022,Jack Ryan,-20


**Check 4: `Email` should contain an "@" symbol and not be blank**

In [12]:
email_stripped = df_deduplicated["Email"].str.strip()
invalid_email = df_deduplicated[
    email_stripped.isna() | (email_stripped == "") | (~email_stripped.str.contains("@", na=False))
]
print(len(invalid_email), "rows with a missing or invalid Email")
invalid_email[["Order ID", "Customer Name", "Email"]]

16 rows with a missing or invalid Email


,Order ID,Customer Name,Email
2,1002,MARY JOHNSON,NaN
4,1004,David Kim,bad-email
5,1005,Bob Lee,NaN
10,1010,Jack Ryan,NaN
11,1011,David Kim,bad-email
12,1012,Paul Young,bad-email
15,1015,Paul Young,NaN
17,1017,jane doe,NaN
18,1018,MARY JOHNSON,NaN
20,1020,Henry Ford,NaN


**Check 5: `Order ID` should be unique (no duplicate IDs) after cleaning**

In [13]:
duplicate_ids_remaining = df_deduplicated.duplicated(subset=["Order ID"]).sum()
print("Duplicate Order IDs remaining:", duplicate_ids_remaining)
print("Order ID uniqueness check passed:", duplicate_ids_remaining == 0)

Duplicate Order IDs remaining: 0
Order ID uniqueness check passed: True


**Check 6: `City` should match one of a known, expected set of cities**

In [14]:
expected_cities = {"los angeles", "new york", "chicago", "miami", "boston", "seattle"}
city_lower = df_deduplicated["City"].str.strip().str.lower()

invalid_city = df_deduplicated[~city_lower.isin(expected_cities) & city_lower.notna()]
print(len(invalid_city), "rows with an unrecognized City value")
print("Actual unique values found:", df_deduplicated["City"].unique())

4 rows with an unrecognized City value
Actual unique values found: <StringArray>
[         'LA',           nan,      'miami ',     'Chicago',    'new york',
     'chicago',      'Boston', 'Los Angeles',    'New York',     'Seattle',
       'Miami']
Length: 11, dtype: str


## 5. Validation Summary

In [15]:
validation_summary = pd.DataFrame({
    "check": [
        "Age out of range (0-120)",
        "Unexpected Gender value",
        "Negative Purchase Amount",
        "Missing/invalid Email",
        "Duplicate Order ID",
        "Unrecognized City"
    ],
    "rows_failed": [
        len(invalid_age),
        len(invalid_gender),
        len(invalid_purchase),
        len(invalid_email),
        duplicate_ids_remaining,
        len(invalid_city)
    ]
})

validation_summary

,check,rows_failed
0,Age out of range (0-120),7
1,Unexpected Gender value,0
2,Negative Purchase Amount,5
3,Missing/invalid Email,16
4,Duplicate Order ID,0
5,Unrecognized City,4


## Outcome

Using `Customer_Dataset.csv`, we found 3 exact duplicate rows and confirmed that the same 3 `Order ID`s were repeated. `drop_duplicates()` removed the extra rows while keeping the first occurrence of each.

We then applied six validation checks to the deduplicated data. They surfaced invalid ages (-5 and 150), inconsistent gender labels, a negative purchase amount, missing or malformed email addresses, and city names with inconsistent casing or abbreviations. The summary table brought the checks together, making it easier to compare the issues and assess data quality than by inspecting rows individually.